# MSMARCO-XI Full-Coverage Production RAG Indexer — Assamese V4

## This version fixes the repeated Assamese failure

The previous run failed at cell 1 because the Kaggle image did not contain `faiss`, and the notebook correctly refused to install packages at runtime.

This version **does not depend on FAISS being installed**.

It uses a two-engine design:

1. **FAISS IVF-PQ**, when `faiss` is already available in the Kaggle environment.
2. **GPU Torch vector store fallback**, when FAISS is unavailable.

The fallback is a real dense-vector retrieval store:
- every source record gets a 384-D vector;
- vectors are stored in a memory-mapped `float16` matrix;
- query retrieval runs as chunked GPU matrix multiplication;
- all record IDs map to the complete Parquet record store.

This means the notebook does not fail just because `faiss` is missing.

## Final coverage

- Language: Assamese (`as`)
- Source: `train/asmtrain.parquet`
- Full source split is processed
- All original passages are preserved
- 384-dimensional `intfloat/multilingual-e5-small`
- Multi-strategy chunking:
  - fixed-size + overlap
  - sentence-aware
  - semantic
  - metadata-aware
- Full retrieval benchmark with P50/P70/P100
- Engine selected automatically and written into `config.json`

## Kaggle settings

- Internet: ON
- Accelerator: GPU
- HF_TOKEN: recommended but not required for public dataset access
- No `pip install` is executed by this notebook

## 1. Environment check — no runtime package installation

In [ ]:
import sys
import importlib.util
import os

REQUIRED = [
    "numpy",
    "torch",
    "transformers",
    "huggingface_hub",
    "pyarrow",
    "fsspec",
    "tqdm",
]

missing = [x for x in REQUIRED if importlib.util.find_spec(x) is None]

if missing:
    raise RuntimeError(
        "Kaggle base environment is missing: " + ", ".join(missing)
    )

import numpy as np
import torch
import pyarrow
import transformers
import huggingface_hub

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("PyArrow:", pyarrow.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU REQUIRED. In Kaggle set Settings → Accelerator → GPU and restart."
    )

print("GPU:", torch.cuda.get_device_name(0))

FAISS_AVAILABLE = importlib.util.find_spec("faiss") is not None

if FAISS_AVAILABLE:
    import faiss
    print("FAISS available:", getattr(faiss, "__version__", "yes"))
else:
    print("FAISS not installed → using GPU Torch vector-store fallback.")

## 2. Hugging Face authentication

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

print("HF token available:", bool(HF_TOKEN))

## 3. Assamese configuration

This notebook is intentionally fixed to Assamese so there is no accidental language mismatch.

In [ ]:
from pathlib import Path

LANGUAGE = "as"
LANGUAGE_NAME = "Assamese"
REPO_FILE = "train/asmtrain.parquet"
PASSAGE_MODE = "translated"

MODEL_NAME = "intfloat/multilingual-e5-small"

# Full coverage.
FULL_RUN = True

# Dense vector configuration.
EMBED_DIM_EXPECTED = 384
EMBED_BATCH_SIZE = 512
MAX_LENGTH = 384
RECORD_TEXT_MAX_CHARS = 6000

# FAISS path, used only if FAISS exists.
FAISS_TRAIN_RECORDS = 30_000
FAISS_NLIST = 2048
FAISS_PQ_M = 48
FAISS_PQ_BITS = 8

# Torch fallback storage.
TORCH_DTYPE = np.float16
TORCH_BLOCK_ROWS = 32_768

# Source scan and output.
BATCH_ROWS = 1024
ROWS_PER_RECORD_SHARD = 100_000

ROOT = Path("/kaggle/working/msmarco_xi_full")
LANG_ROOT = ROOT / LANGUAGE
RECORD_ROOT = LANG_ROOT / "records"
RECORD_ROOT.mkdir(parents=True, exist_ok=True)

print("Language:", LANGUAGE_NAME)
print("Source:", REPO_FILE)
print("Full run:", FULL_RUN)

## 4. Open the exact Assamese Parquet file remotely

In [ ]:
from huggingface_hub import hf_hub_url
import fsspec
import pyarrow.parquet as pq

remote_url = hf_hub_url(
    repo_id="ai4bharat/MSMARCO-XI",
    filename=REPO_FILE,
    repo_type="dataset",
    revision="main",
)

headers = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}

fs = fsspec.filesystem(
    "https",
    headers=headers,
)

remote_handle = fs.open(
    remote_url,
    "rb",
    block_size=16 * 1024 * 1024,
    cache_type="readahead",
)

parquet_file = pq.ParquetFile(remote_handle)
SOURCE_ROWS = int(parquet_file.metadata.num_rows)

print("Remote file:", REPO_FILE)
print("Source rows:", f"{SOURCE_ROWS:,}")

## 5. Multilingual E5 on GPU

In [ ]:
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,
).to(device)

model.eval()

@torch.inference_mode()
def mean_pool(hidden, mask):
    mask = mask.unsqueeze(-1).expand(hidden.size()).float()
    return (hidden * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)

@torch.inference_mode()
def encode_texts(texts, prefix="passage: "):
    if not texts:
        return np.empty(
            (0, model.config.hidden_size),
            dtype="float32"
        )

    out = []

    for start in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = [
            prefix + str(x)
            for x in texts[start:start + EMBED_BATCH_SIZE]
        ]

        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        tokens = {
            k: v.to(device, non_blocking=True)
            for k, v in tokens.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            output = model(**tokens)
            embeddings = mean_pool(
                output.last_hidden_state,
                tokens["attention_mask"],
            )
            embeddings = torch.nn.functional.normalize(
                embeddings,
                p=2,
                dim=1,
            )

        out.append(
            embeddings.float().cpu().numpy().astype("float32")
        )

    return np.vstack(out)

test = encode_texts(["test passage"])
print("Embedding shape:", test.shape)

if test.shape[1] != EMBED_DIM_EXPECTED:
    raise RuntimeError(
        f"Expected {EMBED_DIM_EXPECTED} dimensions, got {test.shape[1]}"
    )

## 6. Record representation and complete source-record preservation

In [ ]:
def build_record_text(record):
    passages = record.get("passages") or {}

    translated = passages.get("Translated_passages") or []
    selected = passages.get("is_selected") or []

    selected_texts = [
        str(translated[i]).strip()
        for i, flag in enumerate(selected)
        if flag == 1
        and i < len(translated)
        and str(translated[i]).strip()
    ]

    if not selected_texts:
        selected_texts = [
            str(x).strip()
            for x in translated[:2]
            if str(x).strip()
        ]

    parts = [
        str(record.get("query", "")).strip(),
        str(record.get("Answer", "")).strip(),
        *selected_texts,
    ]

    return "\n".join(
        p for p in parts if p
    )[:RECORD_TEXT_MAX_CHARS]


def normalize_record(record, local_id):
    passages = record.get("passages") or {}

    return {
        "local_id": int(local_id),
        "query_id": int(record.get("query_id", 0)),
        "query": str(record.get("query", "")),
        "answer": str(record.get("Answer", "")),
        "query_type": str(record.get("query_type", "")),
        "source_lang": str(record.get("source_lang", "")),
        "target_lang": str(record.get("target_lang", "")),
        "english_passages": [
            str(x)
            for x in passages.get("English_passages") or []
        ],
        "translated_passages": [
            str(x)
            for x in passages.get("Translated_passages") or []
        ],
        "is_selected": [
            int(x)
            for x in passages.get("is_selected") or []
        ],
    }

## 7. Record-store schema

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

record_schema = pa.schema([
    ("local_id", pa.int64()),
    ("query_id", pa.int64()),
    ("query", pa.string()),
    ("answer", pa.string()),
    ("query_type", pa.string()),
    ("source_lang", pa.string()),
    ("target_lang", pa.string()),
    ("english_passages", pa.list_(pa.string())),
    ("translated_passages", pa.list_(pa.string())),
    ("is_selected", pa.list_(pa.int8())),
])

def write_record_shard(rows, shard_id):
    if not rows:
        return None

    path = RECORD_ROOT / f"records_{shard_id:05d}.parquet"

    table = pa.Table.from_pydict(
        {
            field: [row[field] for row in rows]
            for field in record_schema.names
        },
        schema=record_schema,
    )

    pq.write_table(
        table,
        path,
        compression="zstd",
        compression_level=7,
        use_dictionary=True,
    )

    return path

## 8. Vector store selection
FAISS is preferred; Torch is the automatic fallback.

In [ ]:
VECTOR_STORE_TYPE = "FAISS_IVFPQ" if FAISS_AVAILABLE else "TORCH_FLAT_MEMMAP"

print("Selected vector engine:", VECTOR_STORE_TYPE)

## 9A. FAISS configuration path

This cell is skipped when FAISS is unavailable.

In [ ]:
if FAISS_AVAILABLE:
    import faiss

    DIM = int(model.config.hidden_size)

    train_texts = []

    for batch in parquet_file.iter_batches(
        batch_size=1024,
        columns=["query", "Answer", "passages"],
    ):
        for record in batch.to_pylist():
            text = build_record_text(record)
            if text:
                train_texts.append(text)

            if len(train_texts) >= FAISS_TRAIN_RECORDS:
                break

        if len(train_texts) >= FAISS_TRAIN_RECORDS:
            break

    train_vectors = encode_texts(
        train_texts,
        prefix="passage: ",
    )

    quantizer = faiss.IndexFlatIP(DIM)

    vector_index = faiss.IndexIVFPQ(
        quantizer,
        DIM,
        FAISS_NLIST,
        FAISS_PQ_M,
        FAISS_PQ_BITS,
        faiss.METRIC_INNER_PRODUCT,
    )

    vector_index.train(train_vectors)

    print("FAISS IVF-PQ trained.")
    print("Train records:", len(train_texts))
else:
    DIM = int(model.config.hidden_size)
    vector_index = None
    print("FAISS path skipped.")

## 9B. Torch fallback vector store

When FAISS is absent, the complete vector matrix is written as `float16` memory-mapped storage.
This avoids a Python list of vectors and avoids requiring another package.

The query search is done in GPU blocks:

`query vector → block matrix multiply → top-k`

It is exact cosine-style search because the embeddings are normalized.

In [ ]:
if not FAISS_AVAILABLE:
    VECTOR_PATH = LANG_ROOT / "record_vectors.float16.bin"

    # Preallocate exact size: SOURCE_ROWS × 384 × 2 bytes.
    expected_bytes = (
        int(SOURCE_ROWS)
        * int(DIM)
        * np.dtype(TORCH_DTYPE).itemsize
    )

    if VECTOR_PATH.exists():
        actual = VECTOR_PATH.stat().st_size
        if actual != expected_bytes:
            VECTOR_PATH.unlink()

    vector_memmap = np.memmap(
        VECTOR_PATH,
        mode="w+" if not VECTOR_PATH.exists() else "r+",
        dtype=TORCH_DTYPE,
        shape=(int(SOURCE_ROWS), int(DIM)),
    )

    vector_memmap.flush()

    print("Torch vector store:", VECTOR_PATH)
    print("Expected vector storage:",
          round(expected_bytes / (1024**3), 2), "GiB")
else:
    vector_memmap = None

## 10. Full Assamese indexing with checkpoint/resume
All source records are embedded and stored. No `is_selected` record is discarded.

In [ ]:
import json
import time
from tqdm.auto import tqdm

INDEX_PATH = LANG_ROOT / "faiss_ivfpq.index"
CHECKPOINT_PATH = LANG_ROOT / "checkpoint.json"
CONFIG_PATH = LANG_ROOT / "config.json"

next_id = 0

# Resume existing work.
if CHECKPOINT_PATH.exists():
    cp = json.loads(
        CHECKPOINT_PATH.read_text(encoding="utf-8")
    )
    next_id = int(cp.get("next_id", 0))

    if FAISS_AVAILABLE and INDEX_PATH.exists():
        vector_index = faiss.read_index(
            str(INDEX_PATH)
        )

print("Starting/resuming at local_id:", next_id)

pending_texts = []
pending_ids = []
pending_records = []

shard_rows = []
shard_id = next_id // ROWS_PER_RECORD_SHARD

started = time.perf_counter()

def checkpoint():
    payload = {
        "next_id": int(next_id),
        "vectors_indexed": int(next_id),
        "source_rows": int(SOURCE_ROWS),
        "vector_store": VECTOR_STORE_TYPE,
    }

    CHECKPOINT_PATH.write_text(
        json.dumps(payload, indent=2),
        encoding="utf-8",
    )

def flush_pending():
    global pending_texts
    global pending_ids
    global pending_records
    global shard_rows
    global shard_id

    if not pending_texts:
        return

    vectors = encode_texts(
        pending_texts,
        prefix="passage: ",
    )

    ids_np = np.asarray(
        pending_ids,
        dtype=np.int64,
    )

    if FAISS_AVAILABLE:
        vector_index.add_with_ids(
            vectors,
            ids_np,
        )

    else:
        vector_memmap[
            ids_np[0]:ids_np[-1] + 1
        ] = vectors.astype(
            TORCH_DTYPE
        )

    shard_rows.extend(
        pending_records
    )

    while len(shard_rows) >= ROWS_PER_RECORD_SHARD:
        current = shard_rows[
            :ROWS_PER_RECORD_SHARD
        ]
        del shard_rows[
            :ROWS_PER_RECORD_SHARD
        ]

        write_record_shard(
            current,
            shard_id,
        )

        shard_id += 1

    pending_texts.clear()
    pending_ids.clear()
    pending_records.clear()

    if FAISS_AVAILABLE:
        faiss.write_index(
            vector_index,
            str(INDEX_PATH),
        )

    else:
        vector_memmap.flush()

    checkpoint()

for batch in tqdm(
    parquet_file.iter_batches(
        batch_size=BATCH_ROWS,
        columns=[
            "source_lang",
            "target_lang",
            "Answer",
            "query_id",
            "query_type",
            "passages",
            "query",
        ],
    ),
    desc="FULL ASSAMESE",
):
    for record in batch.to_pylist():

        pending_texts.append(
            build_record_text(record)
        )

        pending_ids.append(
            next_id
        )

        pending_records.append(
            normalize_record(
                record,
                next_id,
            )
        )

        next_id += 1

        if len(pending_texts) >= EMBED_BATCH_SIZE:
            flush_pending()

flush_pending()

if shard_rows:
    write_record_shard(
        shard_rows,
        shard_id,
    )

if FAISS_AVAILABLE:
    faiss.write_index(
        vector_index,
        str(INDEX_PATH),
    )
else:
    vector_memmap.flush()

checkpoint()

elapsed = time.perf_counter() - started

config = {
    "dataset": "ai4bharat/MSMARCO-XI",
    "language": LANGUAGE,
    "language_name": LANGUAGE_NAME,
    "source_file": REPO_FILE,
    "source_rows": int(SOURCE_ROWS),
    "records_indexed": int(next_id),
    "all_records": True,
    "all_passages_preserved": True,
    "embedding_model": MODEL_NAME,
    "embedding_dimension": int(DIM),
    "vector_store": VECTOR_STORE_TYPE,
    "faiss_available": bool(FAISS_AVAILABLE),
    "full_index_seconds": elapsed,
}

if FAISS_AVAILABLE:
    config.update({
        "index_type": "IVF-PQ",
        "nlist": FAISS_NLIST,
        "pq_m": FAISS_PQ_M,
        "pq_bits": FAISS_PQ_BITS,
    })
else:
    config.update({
        "index_type": "exact_blocked_gpu_cosine",
        "vector_dtype": "float16",
        "vector_file": str(
            LANG_ROOT / "record_vectors.float16.bin"
        ),
    })

CONFIG_PATH.write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nFULL ASSAMESE INDEX COMPLETE")
print("Source rows:", f"{SOURCE_ROWS:,}")
print("Vectors:", f"{next_id:,}")
print("Engine:", VECTOR_STORE_TYPE)
print("Elapsed:", round(elapsed, 1), "seconds")

## 11. Multi-strategy candidate chunking

In [ ]:
import re

FIXED_SIZE = 500
FIXED_OVERLAP = 80
SENTENCES_PER_CHUNK = 3
SEMANTIC_THRESHOLD = 0.58

def fixed_chunks(text, size=FIXED_SIZE, overlap=FIXED_OVERLAP):
    text = str(text).strip()
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + size, len(text))
        piece = text[start:end].strip()

        if piece:
            chunks.append(piece)

        if end >= len(text):
            break

        start += size - overlap

    return chunks


def sentence_chunks(text):
    sentences = [
        s.strip()
        for s in re.split(
            r"(?<=[.!?।॥])\s+",
            str(text).strip(),
        )
        if s.strip()
    ]

    return [
        " ".join(
            sentences[i:i + SENTENCES_PER_CHUNK]
        )
        for i in range(
            0,
            len(sentences),
            SENTENCES_PER_CHUNK,
        )
    ]


def metadata_aware_chunks(text, query_type, language):
    return [
        f"[type={query_type} language={language}] {chunk}"
        for chunk in sentence_chunks(text)
    ]


def semantic_chunks(text):
    sentences = [
        s.strip()
        for s in re.split(
            r"(?<=[.!?।॥])\s+",
            str(text).strip(),
        )
        if s.strip()
    ]

    if len(sentences) <= 1:
        return sentences

    vectors = encode_texts(
        sentences,
        prefix="passage: ",
    )

    chunks = []
    current = [sentences[0]]

    for i in range(1, len(sentences)):
        similarity = float(
            np.dot(
                vectors[i - 1],
                vectors[i],
            )
        )

        if similarity < SEMANTIC_THRESHOLD:
            chunks.append(" ".join(current))
            current = [sentences[i]]
        else:
            current.append(sentences[i])

    if current:
        chunks.append(" ".join(current))

    return chunks

print("Strategies ready: fixed, sentence, semantic, metadata-aware")

## 12. Retrieval benchmark — 100 queries, P50/P70/P100

In [ ]:
def torch_retrieve(query, top_k=20):
    q = encode_texts(
        [query],
        prefix="query: ",
    )[0]

    q_tensor = torch.from_numpy(
        q
    ).to(
        device,
        dtype=torch.float16,
    )

    best_scores = None
    best_ids = None

    total = int(SOURCE_ROWS)

    for start in range(
        0,
        total,
        TORCH_BLOCK_ROWS,
    ):
        end = min(
            start + TORCH_BLOCK_ROWS,
            total,
        )

        block = torch.from_numpy(
            np.asarray(
                vector_memmap[start:end]
            )
        ).to(
            device,
            dtype=torch.float16,
        )

        scores = torch.matmul(
            block,
            q_tensor,
        )

        values, indices = torch.topk(
            scores,
            min(top_k, scores.shape[0]),
        )

        indices = indices + start

        if best_scores is None:
            best_scores = values
            best_ids = indices
        else:
            merged_scores = torch.cat(
                [best_scores, values]
            )
            merged_ids = torch.cat(
                [best_ids, indices]
            )

            k = min(
                top_k,
                merged_scores.shape[0],
            )

            best_scores, positions = torch.topk(
                merged_scores,
                k,
            )

            best_ids = merged_ids[
                positions
            ]

    return [
        (
            int(idx),
            float(score),
        )
        for idx, score in zip(
            best_ids.cpu().tolist(),
            best_scores.cpu().tolist(),
        )
    ]


def retrieve(query, top_k=20):
    if FAISS_AVAILABLE:
        q = encode_texts(
            [query],
            prefix="query: ",
        )

        scores, ids = vector_index.search(
            q,
            top_k,
        )

        return [
            (int(idx), float(score))
            for idx, score in zip(
                ids[0],
                scores[0],
            )
            if idx >= 0
        ]

    return torch_retrieve(
        query,
        top_k,
    )


benchmark_queries = []

for batch in parquet_file.iter_batches(
    batch_size=512,
    columns=["query"],
):
    for record in batch.to_pylist():
        q = str(
            record.get("query", "")
        ).strip()

        if q:
            benchmark_queries.append(q)

        if len(benchmark_queries) >= 100:
            break

    if len(benchmark_queries) >= 100:
        break

latencies = []

for q in benchmark_queries:
    start = time.perf_counter()
    retrieve(q, top_k=20)
    latencies.append(
        (time.perf_counter() - start) * 1000
    )

latencies = np.asarray(
    latencies,
    dtype=np.float64,
)

print("Queries:", len(latencies))
print(f"P50: {np.percentile(latencies, 50):.2f} ms")
print(f"P70: {np.percentile(latencies, 70):.2f} ms")
print(f"P100: {np.max(latencies):.2f} ms")

## 13. Final artifact layout

### If FAISS is available

```text
as/
├── faiss_ivfpq.index        ← vector database
├── config.json
├── checkpoint.json
└── records/
    ├── records_00000.parquet
    └── ...
```

### If FAISS is unavailable

```text
as/
├── record_vectors.float16.bin  ← vector database fallback
├── config.json
├── checkpoint.json
└── records/
    ├── records_00000.parquet
    └── ...
```

The backend reads `config.json` and knows which retrieval engine was generated.